# User-friendly toy generation

This notebook focuses only on pseudo-data generation. Signal-only, efficiency/veto, backgrounds and direct-CP toys all use the same short high-level interface.


In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    CPToyBackground,CPRealImag,DecayChannel,DecayModel,NonResonant,Parameter,
    RealImag,Resonance,ToyBackground,enable_x64,generate_cp_toy,generate_toy,
    plot_dalitz,plot_square_dalitz,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency
enable_x64()


In [ ]:
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [
        Resonance("Kstar",(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),
        NonResonant(RealImag(-0.25,0.10)),
    ],
    normalization_method="square-dalitz",normalization_resolution=160,normalization_pair=(0,2),
)
eff=FunctionalEfficiency(lambda d:0.55+0.30*jnp.clip(d["s13"]/20,0,1))
bkg=FunctionalBackground(lambda d:0.35+0.65*jnp.clip(d["s23"]/25,0,1))

signal=generate_toy(model,15_000,seed=1801)
selected=generate_toy(model,15_000,efficiency=eff,seed=1802)
mixture=generate_toy(
    model,20_000,efficiency=eff,signal_fraction=0.80,
    backgrounds=(ToyBackground("comb",bkg),),
    seed=1803,
)


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4.5),constrained_layout=True)
plot_dalitz(signal,x="s13",y="s23",ax=axes[0],title="signal")
plot_dalitz(selected,x="s13",y="s23",ax=axes[1],title="signal + efficiency")
plot_dalitz(mixture,x="s13",y="s23",ax=axes[2],title="signal + efficiency + background")
plt.show()


In [ ]:
cp=CPRealImag(0.8,0.2,0.08,-0.05)
plus=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),[NonResonant(cp.for_charge(+1))],
    normalization_method="square-dalitz",normalization_resolution=140,normalization_pair=(0,2),
)
minus=DecayModel(
    DecayChannel("B-",("K-","pi-","pi+")),[NonResonant(cp.for_charge(-1))],
    normalization_method="square-dalitz",normalization_resolution=140,normalization_pair=(0,2),
)
plus_toy,minus_toy=generate_cp_toy(
    plus,minus,20_000,plus_efficiency=eff,minus_efficiency=eff,
    signal_fraction=0.85,backgrounds=(CPToyBackground("comb",bkg),),
    seed=1804,
)
print("B+ / B-:",plus_toy.size,minus_toy.size)
